In [2]:
!pip install pandas

In [4]:
import pandas as pd

def generar_muestra_textos():
    # Enlace directo al CSV del dataset de noticias de la BBC en GitHub
    url = "https://raw.githubusercontent.com/suraj-deshmukh/BBC-Dataset-News-Classification/master/dataset/dataset.csv"

    print("Descargando el dataset desde GitHub...")
    try:
        # Leer el archivo CSV con codificación latin-1
        df = pd.read_csv(url, encoding='latin-1')
    except Exception as e:
        print(f"Error al descargar el archivo: {e}")
        return

    print("Procesando y contando palabras...")

    # En este dataset, la columna que contiene el texto se llama 'news'
    # Creamos una nueva columna con el conteo de palabras de cada texto
    df['word_count'] = df['news'].astype(str).apply(lambda x: len(x.split()))

    # Filtramos los textos para que tengan mínimo 100 y máximo 500 palabras
    df_filtrado = df[(df['word_count'] >= 100) & (df['word_count'] <= 500)]

    print(f"Textos que cumplen la condición (100-500 palabras): {len(df_filtrado)}")

    # Verificamos si tenemos al menos 200 textos
    if len(df_filtrado) >= 200:
        # Tomamos una muestra aleatoria de exactamente 200 textos
        # (random_state asegura que siempre obtengas la misma muestra si lo vuelves a ejecutar)
        muestra_final = df_filtrado.sample(n=200, random_state=42)

        nombre_archivo = "200_noticias_ingles.csv"

        # Guardamos solo la columna del texto en un nuevo archivo CSV, ignorando el índice
        muestra_final[['news']].to_csv(nombre_archivo, index=False)

        print("-" * 40)
        print(f"¡Éxito! Se guardó el archivo '{nombre_archivo}' en tu carpeta actual.")
        print("Resumen de tu muestra:")
        print(f"- Cantidad de textos: {len(muestra_final)}")
        print(f"- Promedio de palabras por texto: {muestra_final['word_count'].mean():.1f}")
        print(f"- Texto más corto: {muestra_final['word_count'].min()} palabras")
        print(f"- Texto más largo: {muestra_final['word_count'].max()} palabras")
        print("-" * 40)
    else:
        print("No hay suficientes textos que cumplan los requisitos de longitud.")

# Ejecutar la función
if __name__ == "__main__":
    generar_muestra_textos()

Descargando el dataset desde GitHub...
Procesando y contando palabras...
Textos que cumplen la condición (100-500 palabras): 1761
----------------------------------------
¡Éxito! Se guardó el archivo '200_noticias_ingles.csv' en tu carpeta actual.
Resumen de tu muestra:
- Cantidad de textos: 200
- Promedio de palabras por texto: 294.1
- Texto más corto: 122 palabras
- Texto más largo: 497 palabras
----------------------------------------


In [5]:
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 9.9 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.4.1
    Uninstalling sentence-transformers-5.4.1:
      Successfully uninstalled sentence-transformers-5.4.1


In [7]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import json

def generar_embeddings():
    archivo_entrada = "200_noticias_ingles.csv"

    print("Cargando los textos...")
    try:
        df = pd.read_csv(archivo_entrada)
    except FileNotFoundError:
        print(f"Error: No se encontró '{archivo_entrada}'. Asegúrate de haber ejecutado el script anterior en esta misma sesión de Colab.")
        return

    # 1. Cargar el modelo Transformer
    print("Descargando/Cargando el modelo (puede tardar unos segundos la primera vez)...")
    modelo = SentenceTransformer('all-MiniLM-L6-v2')

    # 2. Convertir los textos a una lista
    textos = df['news'].tolist()

    # 3. Generar los embeddings
    print("Generando embeddings para los 200 textos...")
    # encode() procesa todo automáticamente y devuelve una matriz de numpy
    embeddings = modelo.encode(textos)

    # 4. Preparar la estructura para la Base de Datos
    # Añadimos un ID autoincremental, el texto y el vector asociado
    df_db = pd.DataFrame({
        'id': range(1, len(df) + 1),
        'texto': df['news'],
        'embedding': embeddings.tolist() # Se convierte a lista para que sea compatible con JSON/Bases de datos
    })

    # 5. Guardar los resultados
    # Exportamos a JSON. Es mucho más seguro y estándar que el CSV para importar arreglos de números (vectores) a una BD.
    archivo_salida = "200_noticias_con_embeddings.json"

    # orient='records' genera un formato tipo lista de diccionarios, ideal para MongoDB o APIs de BDs Vectoriales
    df_db.to_json(archivo_salida, orient='records', force_ascii=False, indent=4)

    print("-" * 40)
    print(f"¡Proceso completado exitosamente!")
    print(f"Se guardó el archivo: '{archivo_salida}'")
    print(f"Muestra procesada: {len(df_db)} registros")
    print(f"Dimensión de cada vector (embedding): {len(df_db['embedding'].iloc[0])} dimensiones")
    print("-" * 40)

# Ejecutar el script
generar_embeddings()

Cargando los textos...
Descargando/Cargando el modelo (puede tardar unos segundos la primera vez)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generando embeddings para los 200 textos...
----------------------------------------
¡Proceso completado exitosamente!
Se guardó el archivo: '200_noticias_con_embeddings.json'
Muestra procesada: 200 registros
Dimensión de cada vector (embedding): 384 dimensiones
----------------------------------------
